# BEAD Random Forest — Ablation Study
Log-transformed target with 9 features (AIC-best 8 + jobs_per_location).
Systematically removes one feature at a time to measure each feature's contribution.

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='broadband-data')
print('Connected to BigQuery: broadband-data')

In [ ]:
# Load all data sources
df_projects = client.query("""
SELECT project_id, state, bead_support, estimated_miles_aerial_fiber,
       estimated_miles_buried_fiber, estimated_jobs, project_type, priority_broadband_project
FROM `broadband-data.fp_approved.deployment_projects`
""").to_dataframe()

df_locations = client.query("""
SELECT project_id, COUNT(*) AS funded_locations,
       SAFE_CAST(APPROX_TOP_COUNT(CAST(technology AS STRING), 1)[OFFSET(0)].value AS FLOAT64) AS technology,
       AVG(CAST(low_latency AS INT64)) AS avg_latency
FROM `broadband-data.fp_approved.locations`
GROUP BY project_id
""").to_dataframe()

state_name_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New_Hampshire': 'NH',
    'New_Jersey': 'NJ', 'New_Mexico': 'NM', 'New_York': 'NY', 'North_Carolina': 'NC',
    'North_Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode_Island': 'RI', 'South_Carolina': 'SC', 'South_Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West_Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'District_of_Columbia': 'DC'
}
df_nbm = client.query("""
SELECT state, COUNT(DISTINCT frn) AS state_num_providers
FROM `broadband-data.fcc_nbm.nbm_hive` GROUP BY state
""").to_dataframe()
df_nbm['state'] = df_nbm['state'].map(state_name_to_abbr)
df_nbm = df_nbm.dropna(subset=['state'])

df_state_pop = client.query("""
SELECT stateabbr AS state, SUM(pop2020) AS state_population
FROM `broadband-data.fcc_block_level_pop.us2020` GROUP BY stateabbr
""").to_dataframe()

governor_party_2024 = {
    'AL': 0, 'AK': 0, 'AZ': 1, 'AR': 0, 'CA': 1, 'CO': 1, 'CT': 1, 'DE': 1, 'FL': 0, 'GA': 0,
    'HI': 1, 'ID': 0, 'IL': 1, 'IN': 0, 'IA': 0, 'KS': 1, 'KY': 1, 'LA': 0, 'ME': 1, 'MD': 1,
    'MA': 1, 'MI': 1, 'MN': 1, 'MS': 0, 'MO': 0, 'MT': 0, 'NE': 0, 'NV': 0, 'NH': 0, 'NJ': 1,
    'NM': 1, 'NY': 1, 'NC': 1, 'ND': 0, 'OH': 0, 'OK': 0, 'OR': 1, 'PA': 1, 'RI': 1, 'SC': 0,
    'SD': 0, 'TN': 0, 'TX': 0, 'UT': 0, 'VT': 0, 'VA': 0, 'WA': 1, 'WV': 0, 'WI': 1, 'WY': 0, 'DC': 1
}
df_gov = pd.DataFrame(list(governor_party_2024.items()), columns=['state', 'incumbent_democrat'])

print(f'Projects: {df_projects.shape}, Locations: {df_locations.shape}, NBM: {df_nbm.shape}, Pop: {df_state_pop.shape}')

In [ ]:
# Merge + feature engineering
df = df_projects.merge(df_locations, on='project_id', how='left')
df = df.merge(df_nbm, on='state', how='left')
df = df.merge(df_state_pop, on='state', how='left')
df = df.merge(df_gov, on='state', how='left')

df['funded_locations'] = df['funded_locations'].fillna(0)
df['technology'] = df['technology'].fillna(0)
df['avg_latency'] = df['avg_latency'].fillna(0)
df['state_num_providers'] = df['state_num_providers'].fillna(0)
df['state_population'] = df['state_population'].fillna(0)
df['incumbent_democrat'] = df['incumbent_democrat'].fillna(0).astype(int)

df['total_fiber_miles'] = df['estimated_miles_aerial_fiber'].fillna(0) + df['estimated_miles_buried_fiber'].fillna(0)
df['miles_per_location'] = df['total_fiber_miles'] / df['funded_locations'].replace(0, np.nan)
df['miles_per_location'] = df['miles_per_location'].fillna(0)
df['jobs_per_location'] = df['estimated_jobs'] / df['funded_locations'].replace(0, np.nan)
df['jobs_per_location'] = df['jobs_per_location'].fillna(0)
df['priority_broadband_project'] = df['priority_broadband_project'].map({True: 1, False: 0, 'Yes': 1, 'No': 0, 1: 1, 0: 0}).fillna(0).astype(int)

df['funding_per_location'] = df['bead_support'] / df['funded_locations'].replace(0, np.nan)
df = df.dropna(subset=['funding_per_location'])
low = df['funding_per_location'].quantile(0.025)
high = df['funding_per_location'].quantile(0.975)
df = df[(df['funding_per_location'] >= low) & (df['funding_per_location'] <= high)]

df['log_funding'] = np.log1p(df['funding_per_location'])
print(f'Samples: {df.shape[0]}')

In [ ]:
# Baseline: all 9 features
all_features = [
    'technology', 'avg_latency', 'total_fiber_miles', 'miles_per_location',
    'jobs_per_location', 'priority_broadband_project',
    'state_num_providers', 'state_population', 'incumbent_democrat',
]

X = df[all_features].fillna(0)
y = df['log_funding']
kf = KFold(n_splits=5, shuffle=True, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
rf_base = RandomForestRegressor(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
yp_base = rf_base.predict(X_test)

r2_base = r2_score(y_test, yp_base)
cv_base = cross_val_score(rf_base, X, y, cv=kf, scoring='r2')
rmse_base = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(yp_base)))
mae_base = mean_absolute_error(np.expm1(y_test), np.expm1(yp_base))

print('=== Baseline: All 9 Features ===')
print(f'Test R² (log):   {r2_base:.4f}')
print(f'CV R² (log):     {cv_base.mean():.4f} +/- {cv_base.std():.4f}')
print(f'RMSE (real $):   {rmse_base:,.0f}')
print(f'MAE (real $):    {mae_base:,.0f}')

In [ ]:
# Ablation: remove one feature at a time
ablation_results = []

for drop_feat in all_features:
    feats = [f for f in all_features if f != drop_feat]
    X_ab = df[feats].fillna(0)
    
    X_tr, X_te, y_tr, y_te = train_test_split(X_ab, y, test_size=0.3, random_state=42)
    rf_ab = RandomForestRegressor(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1)
    rf_ab.fit(X_tr, y_tr)
    yp_ab = rf_ab.predict(X_te)
    
    r2_ab = r2_score(y_te, yp_ab)
    cv_ab = cross_val_score(rf_ab, X_ab, y, cv=kf, scoring='r2')
    rmse_ab = np.sqrt(mean_squared_error(np.expm1(y_te), np.expm1(yp_ab)))
    mae_ab = mean_absolute_error(np.expm1(y_te), np.expm1(yp_ab))
    
    ablation_results.append({
        'Dropped': drop_feat,
        'Test R²': r2_ab,
        'CV R²': cv_ab.mean(),
        'R² Drop': r2_base - r2_ab,
        'RMSE ($)': rmse_ab,
        'MAE ($)': mae_ab,
    })

ab_df = pd.DataFrame(ablation_results).sort_values('R² Drop', ascending=False)
ab_df['Test R²'] = ab_df['Test R²'].round(4)
ab_df['CV R²'] = ab_df['CV R²'].round(4)
ab_df['R² Drop'] = ab_df['R² Drop'].round(4)
ab_df['RMSE ($)'] = ab_df['RMSE ($)'].apply(lambda x: f'{x:,.0f}')
ab_df['MAE ($)'] = ab_df['MAE ($)'].apply(lambda x: f'{x:,.0f}')

print(f'Baseline R²: {r2_base:.4f}')
print()
print(ab_df.to_string(index=False))

In [ ]:
# Ablation bar chart — R² drop when each feature is removed
ab_plot = pd.DataFrame(ablation_results).sort_values('R² Drop', ascending=True)

colors = ['crimson' if d > 0.005 else 'steelblue' if d > 0 else 'gray' for d in ab_plot['R² Drop']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(ab_plot['Dropped'], ab_plot['R² Drop'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('R² Drop (higher = more important)')
ax.set_title(f'Ablation Study — Feature Contribution to R² (baseline={r2_base:.4f})')
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative ablation: add features one at a time in order of importance
importance_order = pd.DataFrame(ablation_results).sort_values('R² Drop', ascending=False)['Dropped'].tolist()

cum_results = []
for i in range(1, len(importance_order) + 1):
    feats = importance_order[:i]
    X_cum = df[feats].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(X_cum, y, test_size=0.3, random_state=42)
    rf_cum = RandomForestRegressor(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1)
    rf_cum.fit(X_tr, y_tr)
    yp_cum = rf_cum.predict(X_te)
    r2_cum = r2_score(y_te, yp_cum)
    cv_cum = cross_val_score(rf_cum, X_cum, y, cv=kf, scoring='r2').mean()
    cum_results.append({'k': i, 'features': ', '.join(feats), 'Test R²': r2_cum, 'CV R²': cv_cum})
    print(f'k={i}  R²={r2_cum:.4f}  CV={cv_cum:.4f}  +{feats[-1]}')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ks = [r['k'] for r in cum_results]
r2s = [r['Test R²'] for r in cum_results]
cvs = [r['CV R²'] for r in cum_results]
ax.plot(ks, r2s, 'o-', color='steelblue', label='Test R²')
ax.plot(ks, cvs, 's--', color='coral', label='CV R²')
ax.set_xlabel('Number of Features (added in ablation-importance order)')
ax.set_ylabel('R²')
ax.set_title('Cumulative Feature Addition — Diminishing Returns')
ax.set_xticks(ks)
ax.set_xticklabels([importance_order[i-1] for i in ks], rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from baseline model
imp_df = pd.DataFrame({
    'feature': all_features,
    'rf_importance': rf_base.feature_importances_,
    'ablation_r2_drop': [r['R² Drop'] for r in sorted(ablation_results, key=lambda x: all_features.index(x['Dropped']))]
}).sort_values('ablation_r2_drop', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_sorted = imp_df.sort_values('rf_importance', ascending=True)
axes[0].barh(imp_sorted['feature'], imp_sorted['rf_importance'], color='steelblue')
axes[0].set_title('RF Feature Importance (Gini)')
axes[0].set_xlabel('Importance')

ab_sorted = imp_df.sort_values('ablation_r2_drop', ascending=True)
axes[1].barh(ab_sorted['feature'], ab_sorted['ablation_r2_drop'], color='coral')
axes[1].set_title('Ablation R² Drop')
axes[1].set_xlabel('R² Drop')

plt.suptitle('Feature Importance: Gini vs Ablation', y=1.02)
plt.tight_layout()
plt.show()

print(imp_df[['feature', 'rf_importance', 'ablation_r2_drop']].to_string(index=False))